# Pilot v3 Analysis — reading conditions

# Conditions (assigned_doc): birds, birds_listed, birds_repeat, birds_distractors, birds_easier
# Date filter: June 18 2026 only
# Scoring:
#   all-or-nothing (AoN): 1 if selected set == answer set exactly, else 0
#   edit distance: |selected Δ answer| — number of option-level errors (0 = perfect, 5 = worst)

In [ ]:
import csv
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml

HERE      = Path('/Users/cl5625/simulating-memory/.claude/worktrees/compare-task/application/prolific_study/pilot_v3')
YAML_PATH = HERE.parent / 'birds_q_multiselect.yaml'
CSV_PATH  = HERE / 'results.csv'

DATE_FILTER = '2026-06-18'

QCOLS      = [f'Q{n:02d}' for n in range(1, 11)]
QIDS       = [f'QS{n:02d}' for n in range(1, 11)]
COL_TO_QID = dict(zip(QCOLS, QIDS))

CONDITIONS = ['birds', 'birds_listed', 'birds_repeat', 'birds_distractors', 'birds_easier']
COND_COLORS = {
    'birds':            '#1f77b4',
    'birds_listed':     '#ff7f0e',
    'birds_repeat':     '#2ca02c',
    'birds_distractors': '#d62728',
    'birds_easier':     '#9467bd',
}
COND_LABELS = {
    'birds':            'birds (control)',
    'birds_listed':     'birds_listed',
    'birds_repeat':     'birds_repeat',
    'birds_distractors': 'birds_distractors',
    'birds_easier':     'birds_easier',
}

AT1_CORRECT = 'Green versus rose'
AT2_CORRECT = 'Plumage strategy'

In [ ]:
# Load YAML — options and answers are integer-keyed (1-5)
spec   = yaml.safe_load(YAML_PATH.read_text())
by_qid = {q['q_id']: q for q in spec['questions']}

In [ ]:
# Load CSV, apply filters
with CSV_PATH.open(newline='', encoding='utf-8-sig') as f:
    raw_rows = list(csv.DictReader(f))

all_rows = raw_rows[2:]  # skip label + importId header rows
finished = [r for r in all_rows if r.get('Finished', '').lower() in {'true', '1'}]
date_ok  = [r for r in finished if r.get('RecordedDate', '').startswith(DATE_FILTER)]
kept_by_cond = {}
for cond in CONDITIONS:
    cond_rows = [r for r in date_ok if r.get('assigned_doc', '') == cond]
    kept = [
        r for r in cond_rows
        if r.get('PROLIFIC_PID', '').strip()
        and r.get('AT1', '').strip() == AT1_CORRECT
        and r.get('AT2', '').strip() == AT2_CORRECT
    ]
    kept_by_cond[cond] = kept
    print(f'{cond}: date_ok={len(cond_rows)}, kept after AT exclusions={len(kept)}')

In [ ]:
# OPTION_ALIASES: maps (qid, option_key_int) -> list of survey text variants
# Only needed where survey text diverges from YAML option text.
# Note: parse_selected already strips markdown underscores (_), so QS09 italic names match without aliases.
OPTION_ALIASES = {
    # QS06 option 1: YAML 'named after', survey 'named for'
    ('QS06', 1): ['The sunwhistle is named for its translucent wing feathers.'],
}


def parse_selected(response_text, options_dict, qid=None, strict=False):
    if not response_text or not response_text.strip():
        return set()
    resp = re.sub(r' -- ', ' - ', response_text.strip())
    selected = set()
    for key_int, opt_text in options_dict.items():
        canon = re.sub(r'_', '', opt_text.strip()).rstrip('.')
        canon = re.sub(r' -- ', ' - ', canon)
        if canon in resp:
            selected.add(key_int)
        if qid:
            for alias in OPTION_ALIASES.get((qid, key_int), []):
                if re.sub(r'_', '', alias.strip()).rstrip('.') in resp:
                    selected.add(key_int)
    if strict and not selected:
        raise ValueError(
            f'[{qid}] Response matched no option.\n'
            f'  Response: {repr(response_text[:120])}\n'
            f'  Options:  {list(options_dict.values())}'
        )
    return selected


def all_or_nothing(selected, answer_key):
    return int(set(selected) == set(answer_key))


def edit_distance(selected, answer_key):
    """Number of option-level errors: |selected Δ answer|. Range 0–n_opts. Lower = better."""
    return len(set(selected).symmetric_difference(set(answer_key)))


respondent_data = []
for cond, rows in kept_by_cond.items():
    for r in rows:
        rec = {
            'cond': cond,
            'pid':  (r.get('PROLIFIC_PID') or r.get('ResponseId', ''))[:12],
        }
        for qcol, qid in COL_TO_QID.items():
            q   = by_qid[qid]
            sel = parse_selected(r.get(qcol, ''), q['options'], qid=qid, strict=True)
            rec[qid] = {
                'selected':  sel,
                'aon':       all_or_nothing(sel, q['answer']),
                'edit_dist': edit_distance(sel, q['answer']),
            }
        rec['total_aon']       = sum(rec[qid]['aon']       for qid in QIDS)
        rec['total_edit_dist'] = np.mean([rec[qid]['edit_dist'] for qid in QIDS])
        respondent_data.append(rec)

PARAPHRASE = {qid for qid in QIDS if by_qid[qid]['metadata'].get('cue_match') == 'paraphrase'}

print(f'Parsed {len(respondent_data)} respondents — strict mode passed.')
print(f'Paraphrase questions: {PARAPHRASE}')

In [ ]:
# ── Shared layout helpers ──────────────────────────────────────────────────────
n_cond  = len(CONDITIONS)
bar_w   = 0.7 / n_cond
offsets = np.linspace(-(n_cond - 1) / 2, (n_cond - 1) / 2, n_cond) * bar_w
x       = np.arange(len(QIDS))


def wrap(s, n=22):
    s = re.sub(r'_', '', s.strip())
    words, lines, line = s.split(), [], ''
    for w in words:
        if len(line) + len(w) + 1 > n: lines.append(line); line = w
        else: line = (line + ' ' + w).strip()
    if line: lines.append(line)
    return '\n'.join(lines)

In [ ]:
# ── Answer distributions: one file per question ────────────────────────────────

for qcol, qid in COL_TO_QID.items():
    q        = by_qid[qid]
    opts     = q['options']            # {1: text, 2: text, ...}
    ans_set  = set(q['answer'])
    opt_keys = list(opts.keys())       # [1, 2, 3, 4, 5]
    xq       = np.arange(len(opt_keys))
    star     = '*' if qid in PARAPHRASE else ''

    fig, ax = plt.subplots(figsize=(9, 5))

    for i, k in enumerate(opt_keys):
        if k in ans_set:
            ax.axvspan(i - 0.5, i + 0.5, color='#fffacd', zorder=0)

    for ci, cond in enumerate(CONDITIONS):
        recs  = [rec for rec in respondent_data if rec['cond'] == cond]
        n     = len(recs)
        rates = [
            sum(1 for rec in recs if k in rec[qid]['selected']) / n * 100 if n else 0
            for k in opt_keys
        ]
        ax.bar(xq + offsets[ci], rates, width=bar_w * 0.9,
               color=COND_COLORS[cond], label=f'{COND_LABELS[cond]} (N={n})', zorder=2)

    ax.set_title(f'{qid}{star} ({qcol})  |  answer: {q["answer"]}\n{q["question"]}',
                 fontsize=10, loc='left')
    ax.set_xticks(xq)
    ax.set_xticklabels([f'{k}\n{wrap(opts[k])}' for k in opt_keys], fontsize=7)
    ax.set_ylim(0, 110)
    ax.set_ylabel('% selected', fontsize=9)
    ax.tick_params(axis='y', labelsize=8)
    ax.legend(fontsize=7, ncol=n_cond, loc='upper right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.text(0.01, 0.97, 'yellow = correct option  |  * = paraphrase cue-match',
            transform=ax.transAxes, fontsize=7, va='top', color='#555')

    plt.tight_layout()
    out = HERE / f'answer_dist_{qid}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {out.name}')

In [ ]:
# ── Correctness grids: 1 row × 5 cols ─────────────────────────────────────────

from matplotlib.colors import ListedColormap
cmap_pad = ListedColormap(['#ffffff', '#eeeeee', '#2ca02c'])

max_n = max((len(kept_by_cond[c]) for c in CONDITIONS), default=1)
fig, axes = plt.subplots(1, n_cond, figsize=(3 * n_cond, max(4, 0.42 * max_n)))

for ax, cond in zip(axes, CONDITIONS):
    recs = sorted(
        [rec for rec in respondent_data if rec['cond'] == cond],
        key=lambda r: -r['total_aon']
    )
    n = len(recs)
    if n == 0:
        ax.axis('off')
        continue

    M        = np.array([[rec[qid]['aon'] for qid in QIDS] for rec in recs])
    row_accs = M.mean(axis=1)
    col_accs = M.mean(axis=0)
    pids     = [r['pid'] or f'R{i}' for i, r in enumerate(recs)]

    pad   = np.full((max_n - n, len(QIDS)), -1) if n < max_n else np.empty((0, len(QIDS)))
    M_pad = np.vstack([M, pad]) if len(pad) else M

    ax.imshow(M_pad, aspect='auto', cmap=cmap_pad, vmin=-1, vmax=1, interpolation='nearest')

    xlabels = [f'{qid}{"*" if qid in PARAPHRASE else ""}\n({a:.0%})' for qid, a in zip(QIDS, col_accs)]
    ax.set_xticks(range(len(QIDS)))
    ax.set_xticklabels(xlabels, fontsize=5.5, rotation=45, ha='right')
    ylabels = [f'{p} {a:.0%}' for p, a in zip(pids, row_accs)] + [''] * max(0, max_n - n)
    ax.set_yticks(range(max_n))
    ax.set_yticklabels(ylabels, fontsize=5.5)
    ax.set_title(f'{COND_LABELS[cond]}\n(N={n})', fontsize=8,
                 color=COND_COLORS[cond], fontweight='bold')

    ax.set_xticks(np.arange(-0.5, len(QIDS), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, max_n, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=1)
    ax.tick_params(which='minor', length=0)

plt.suptitle('Correctness grids by condition  |  green = correct (all-or-nothing)  |  * = paraphrase',
             fontsize=10, y=1.02)
plt.tight_layout()
out = HERE / 'correctness_grids.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved {out.name}')

In [ ]:
# ── Score comparison: one row per condition ────────────────────────────────────

fig, axes = plt.subplots(n_cond, 2, figsize=(10, 2.5 * n_cond), sharex='col')

bins_aon  = np.arange(-0.5, len(QIDS) + 1.5, 1)
bins_edit = np.arange(-0.5, 5.5 + 0.01, 0.5)

for row, cond in enumerate(CONDITIONS):
    recs  = [rec for rec in respondent_data if rec['cond'] == cond]
    n     = len(recs)
    color = COND_COLORS[cond]
    label = COND_LABELS[cond]

    aon_scores  = [rec['total_aon']       for rec in recs]
    edit_scores = [rec['total_edit_dist'] for rec in recs]

    ax = axes[row, 0]
    if aon_scores:
        ax.hist(aon_scores, bins=bins_aon, color=color, alpha=0.85)
        ax.axvline(np.mean(aon_scores), color='black', linestyle='--', linewidth=1.2)
        ax.text(0.97, 0.95, f'μ={np.mean(aon_scores):.1f}', transform=ax.transAxes,
                ha='right', va='top', fontsize=8)
    ax.set_ylabel(f'{label}\n(N={n})', fontsize=8, color=color, fontweight='bold')
    if row == 0:
        ax.set_title('All-or-nothing (of 10)', fontsize=9)
    if row == n_cond - 1:
        ax.set_xlabel('# correct', fontsize=8)
    ax.set_xlim(-0.5, len(QIDS) + 0.5)

    ax = axes[row, 1]
    if edit_scores:
        ax.hist(edit_scores, bins=bins_edit, color=color, alpha=0.85)
        ax.axvline(np.mean(edit_scores), color='black', linestyle='--', linewidth=1.2)
        ax.text(0.97, 0.95, f'μ={np.mean(edit_scores):.2f}', transform=ax.transAxes,
                ha='right', va='top', fontsize=8)
    if row == 0:
        ax.set_title('Mean edit distance (lower = better)', fontsize=9)
    if row == n_cond - 1:
        ax.set_xlabel('Edit distance', fontsize=8)

fig.suptitle('Score distributions by condition  |  dashed = mean', fontsize=11, y=1.01)
plt.tight_layout()
out = HERE / 'score_comparison.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved {out.name}')

In [ ]:
# ── Per-question accuracy and edit distance ────────────────────────────────────

def bootstrap_ci(vals, n_boot=2000, ci=95, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    vals = np.asarray(vals, dtype=float)
    if len(vals) == 0:
        return 0.0, 0.0
    boot_means = np.array([rng.choice(vals, size=len(vals), replace=True).mean()
                           for _ in range(n_boot)])
    lo = np.percentile(boot_means, (100 - ci) / 2)
    hi = np.percentile(boot_means, 100 - (100 - ci) / 2)
    return vals.mean() - lo, hi - vals.mean()


rng_boot = np.random.default_rng(42)


def bar_plot(metric, ylabel, fname, ylim, title, scale=1.0):
    fig, ax = plt.subplots(figsize=(14, 6))
    for ci, cond in enumerate(CONDITIONS):
        recs  = [rec for rec in respondent_data if rec['cond'] == cond]
        n     = len(recs)
        means, lo_errs, hi_errs = [], [], []
        for qid in QIDS:
            vals = np.array([rec[qid][metric] for rec in recs]) * scale
            m    = vals.mean() if n else 0
            lo, hi = bootstrap_ci(vals, rng=rng_boot)
            means.append(m); lo_errs.append(lo); hi_errs.append(hi)
        ax.bar(x + offsets[ci], means, width=bar_w * 0.9,
               color=COND_COLORS[cond], label=f'{COND_LABELS[cond]} (N={n})', zorder=2)
        ax.errorbar(x + offsets[ci], means, yerr=[lo_errs, hi_errs],
                    fmt='none', ecolor='#444', elinewidth=1, capsize=3, capthick=1, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{qid}{"*" if qid in PARAPHRASE else ""}' for qid in QIDS], fontsize=10)
    ax.set_xlim(-0.6, len(QIDS) - 0.4)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_ylim(*ylim)
    ax.set_title(f'{title}\nerror bars = 95% bootstrap CI  |  * = paraphrase', fontsize=10)
    ax.legend(fontsize=8, ncol=n_cond, loc='upper right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    out = HERE / fname
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {out.name}')


bar_plot('aon',       '% correct (all-or-nothing)',          'per_question_accuracy.png',  (0, 110), 'Per-question accuracy by condition',      scale=100)
bar_plot('edit_dist', 'Mean edit distance (lower = better)', 'per_question_edit_dist.png', (0, 5),   'Per-question edit distance by condition', scale=1)

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────────
print(f'{"Condition":<22} {"N":>4}  {"Mean AoN /10":>12}  {"Mean edit dist":>14}')
print('-' * 56)
for cond in CONDITIONS:
    recs = [rec for rec in respondent_data if rec['cond'] == cond]
    n    = len(recs)
    if not recs:
        continue
    mean_aon  = np.mean([r['total_aon']       for r in recs])
    mean_edit = np.mean([r['total_edit_dist'] for r in recs])
    print(f'{COND_LABELS[cond]:<22} {n:>4}  {mean_aon:>12.2f}  {mean_edit:>14.3f}')

print(f'\nEdit distance = |selected Δ answer| per question, averaged over {len(QIDS)} questions.')
print('Range 0 (perfect) – 5 (worst). Lower = better.')

In [ ]:
# ── Reported difficulty by condition ──────────────────────────────────────────
# Q34 = passage difficulty (0=easy, 10=hard)
# Q23 = question difficulty (0=easy, 10=hard)

diff_cols = {
    'Passage difficulty (Q34)':  'Q34',
    'Question difficulty (Q23)': 'Q23',
}

diff_data = {label: {cond: [] for cond in CONDITIONS} for label in diff_cols}
for cond, rows in kept_by_cond.items():
    for r in rows:
        for label, col in diff_cols.items():
            val = r.get(col, '').strip()
            if val:
                try:
                    diff_data[label][cond].append(float(val))
                except ValueError:
                    pass

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
rng_jitter = np.random.default_rng(42)

for ax, (label, col) in zip(axes, diff_cols.items()):
    for ci, cond in enumerate(CONDITIONS):
        vals = np.array(diff_data[label][cond])
        if len(vals) == 0:
            continue
        color = COND_COLORS[cond]

        ax.boxplot(vals, positions=[ci], widths=0.5,
                   patch_artist=True, zorder=2,
                   boxprops=dict(facecolor=color, alpha=0.35, linewidth=1.2),
                   medianprops=dict(color=color, linewidth=2),
                   whiskerprops=dict(color=color, linewidth=1.2),
                   capprops=dict(color=color, linewidth=1.2),
                   flierprops=dict(marker='', linestyle='none'))

        jitter = rng_jitter.uniform(-0.18, 0.18, size=len(vals))
        ax.scatter(ci + jitter, vals, color=color, alpha=0.7, s=25, zorder=3)
        ax.scatter(ci, vals.mean(), marker='D', color=color, s=50,
                   edgecolors='white', linewidths=0.8, zorder=4)

    ax.set_xticks(range(len(CONDITIONS)))
    ax.set_xticklabels([COND_LABELS[c] for c in CONDITIONS], fontsize=8, rotation=15, ha='right')
    ax.set_title(label, fontsize=10)
    ax.set_ylabel('Difficulty (0=easy, 10=hard)', fontsize=9)
    ax.set_ylim(-0.5, 10.5)
    ax.set_yticks(range(11))
    ax.tick_params(axis='y', labelsize=8)

    for ci, cond in enumerate(CONDITIONS):
        vals = np.array(diff_data[label][cond])
        if len(vals):
            ax.text(ci, vals.mean() + 0.45, f'{vals.mean():.1f}',
                    ha='center', fontsize=7.5, color=COND_COLORS[cond], fontweight='bold')

plt.suptitle('Reported difficulty by condition  |  diamond=mean  |  box=IQR',
             fontsize=10, y=1.02)
plt.tight_layout()
out = HERE / 'reported_difficulty.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved {out.name}')

In [ ]:
# ── Per-respondent accuracy (all-or-nothing) ───────────────────────────────────

from matplotlib.patches import Patch

sorted_recs = sorted(respondent_data, key=lambda r: -r['total_aon'])
accs   = [r['total_aon'] / len(QIDS) * 100 for r in sorted_recs]
colors = [COND_COLORS[r['cond']] for r in sorted_recs]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(sorted_recs)), accs, color=colors, edgecolor='white', linewidth=0.4)

legend_handles = [Patch(facecolor=COND_COLORS[c], label=COND_LABELS[c]) for c in CONDITIONS]
ax.legend(handles=legend_handles, fontsize=8, ncol=len(CONDITIONS), loc='upper right')

ax.set_xlabel('Respondent (sorted by accuracy)', fontsize=9)
ax.set_ylabel('% correct (all-or-nothing)', fontsize=9)
ax.set_ylim(0, 110)
ax.set_xticks([])
ax.set_title(f'Per-respondent accuracy  (N={len(sorted_recs)}, {len(QIDS)} questions)', fontsize=10)
if accs:
    ax.axhline(np.mean(accs), color='black', linestyle='--', linewidth=1,
               label=f'mean={np.mean(accs):.0f}%')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for i, (bar_x, acc) in enumerate(zip(range(len(sorted_recs)), accs)):
    ax.text(bar_x, acc + 1, f'{acc:.0f}',
            ha='center', va='bottom', fontsize=6.5, rotation=90)

plt.tight_layout()
out = HERE / 'per_respondent_accuracy.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved {out.name}')